## Part 1: Start Spark


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark = SparkSession.builder.appName("Customer Churn Capstone").getOrCreate()
print("Spark version:", spark.version)


Spark version: 4.1.0


## Part 2: Load Data


### Task 1: Load CSV

In [0]:
file_path =spark.table("workspace.default.customer_churn")
#TODO: Write code to read the csv dataset, set header to be true and inferSchema to be true
customers_df = file_path
customers_df.show(5)

+-----------+---+-------------+-------------+----------------+-----------------+-------------+-------------+---------+------------------+-------+
|customer_id|age|annual_income|monthly_spend|visits_per_month|years_as_customer|support_calls|discount_used|   region|preferred_category|churned|
+-----------+---+-------------+-------------+----------------+-----------------+-------------+-------------+---------+------------------+-------+
|          1| 58|        89691|       270.03|               7|                1|            3|            1|  Midwest|         Education|      1|
|          2| 23|        67110|        278.9|               8|               10|            2|            1|Northeast|          Business|      0|
|          3| 35|        62916|       170.96|               8|                5|            0|            0|    South|          Business|      0|
|          4| 39|        79223|       229.77|               5|                4|            3|            0|Northeast|      

### Task 2: Inspect the data



In [0]:

customers_df.printSchema()
customers_df.describe().show()

root
 |-- customer_id: long (nullable = true)
 |-- age: long (nullable = true)
 |-- annual_income: long (nullable = true)
 |-- monthly_spend: double (nullable = true)
 |-- visits_per_month: long (nullable = true)
 |-- years_as_customer: long (nullable = true)
 |-- support_calls: long (nullable = true)
 |-- discount_used: long (nullable = true)
 |-- region: string (nullable = true)
 |-- preferred_category: string (nullable = true)
 |-- churned: long (nullable = true)

+-------+-----------------+------------------+-----------------+------------------+------------------+-----------------+----------------+------------------+-------+------------------+-------------------+
|summary|      customer_id|               age|    annual_income|     monthly_spend|  visits_per_month|years_as_customer|   support_calls|     discount_used| region|preferred_category|            churned|
+-------+-----------------+------------------+-----------------+------------------+------------------+-----------------+

In [0]:
# Step 1: 
customers_df.select(
    "annual_income",
    F.col("annual_income").isNull().alias("missing")
).show(10)

+-------------+-------+
|annual_income|missing|
+-------------+-------+
|        89691|  false|
|        67110|  false|
|        62916|  false|
|        79223|  false|
|        78126|  false|
|        71225|  false|
|        66364|  false|
|        93379|  false|
|        60315|  false|
|        45314|  false|
+-------------+-------+
only showing top 10 rows


In [0]:
# Step 2: 
customers_df.select(
    "annual_income",
    F.when(
        F.col("annual_income").isNull(),
        1
    ).otherwise(0).alias("missing")
).show(10)

+-------------+-------+
|annual_income|missing|
+-------------+-------+
|        89691|      0|
|        67110|      0|
|        62916|      0|
|        79223|      0|
|        78126|      0|
|        71225|      0|
|        66364|      0|
|        93379|      0|
|        60315|      0|
|        45314|      0|
+-------------+-------+
only showing top 10 rows


In [0]:
#Step 3
# TODO: Now you can write code to count the number of 1's in the column annual_income. 
customers_df.select(
    F.count(F.when(F.col("annual_income") == 1, 1)).alias("missing")
).show()

+-------+
|missing|
+-------+
|      0|
+-------+



In [0]:
# TODO: Count missing values in each column
from pyspark.sql import functions as F

customers_df.select([
    F.count(
        F.when(
            F.col(c).isNull(),
           c
        )
    ).alias(c)
    for c in customers_df.columns
]).show()

+-----------+---+-------------+-------------+----------------+-----------------+-------------+-------------+------+------------------+-------+
|customer_id|age|annual_income|monthly_spend|visits_per_month|years_as_customer|support_calls|discount_used|region|preferred_category|churned|
+-----------+---+-------------+-------------+----------------+-----------------+-------------+-------------+------+------------------+-------+
|          0|  0|            0|            0|               0|                0|            0|            0|     0|                 0|      0|
+-----------+---+-------------+-------------+----------------+-----------------+-------------+-------------+------+------------------+-------+



### Task 3.2: Check Duplicate customer_id values. 


In [0]:
# TODO: Check duplicate customer_id values. 
# First, we group the rows by customer_id. Next, we count how many records belong to each customer ID. Finally, we keep only those customer IDs whose count is greater than one (hint: use filter()). If nothing is returned, then every customer ID is unique and there are no duplicates.
customers_df.groupBy("customer_id").count().filter(F.col("count") > 1).show()


+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



## Part 3: DataFrame analysis


In [0]:
# TODO: Create spend_per_visit and display 10 rows
# Hint 1
# To create a new column in a DataFrame, use the withColumn() method.
customers_df.withColumn("spend_per_visit", F.round(F.col("monthly_spend") / F.col("visits_per_month"), 2)).show(10)

+-----------+---+-------------+-------------+----------------+-----------------+-------------+-------------+---------+------------------+-------+---------------+
|customer_id|age|annual_income|monthly_spend|visits_per_month|years_as_customer|support_calls|discount_used|   region|preferred_category|churned|spend_per_visit|
+-----------+---+-------------+-------------+----------------+-----------------+-------------+-------------+---------+------------------+-------+---------------+
|          1| 58|        89691|       270.03|               7|                1|            3|            1|  Midwest|         Education|      1|          38.58|
|          2| 23|        67110|        278.9|               8|               10|            2|            1|Northeast|          Business|      0|          34.86|
|          3| 35|        62916|       170.96|               8|                5|            0|            0|    South|          Business|      0|          21.37|
|          4| 39|        792

### Task 5: Customer summary by region



In [0]:
# TODO: Create regional summary
# Final Template; you can just modify the following; again, you just need to submit your final code
customers_df.groupBy("region") \
    .agg(
        F.count("*").alias("customer_count"),
        F.round(F.avg("monthly_spend"), 2).alias("avg_monthly_spend"),
        F.round(F.avg("visits_per_month"), 2).alias("avg_visits"),
        F.round(F.avg("churned"), 2).alias("churn_rate")
    ) \
    .orderBy(F.desc("churn_rate")) \
    .show()


+---------+--------------+-----------------+----------+----------+
|   region|customer_count|avg_monthly_spend|avg_visits|churn_rate|
+---------+--------------+-----------------+----------+----------+
|  Midwest|           124|           180.08|      6.53|      0.22|
|    South|           112|           178.81|       6.6|      0.18|
|Northeast|           135|           178.03|      6.01|      0.17|
|     West|           129|           175.89|      6.91|      0.14|
+---------+--------------+-----------------+----------+----------+



## Part 4: Spark SQL analysis


In [0]:
customers_df.createOrReplaceTempView("customers")


### Task 6: Churn by preferred category



In [0]:
spark.sql("""
    SELECT preferred_category AS category, 
        COUNT(CUSTOMER_ID) AS customer_count, 
        ROUND(AVG(monthly_spend), 2) AS avg_monthly_spend, 
        ROUND(AVG(churned), 2) AS churn_rate
    FROM customers
    GROUP BY category
    ORDER BY churn_rate DESC
""").show()


+----------+--------------+-----------------+----------+
|  category|customer_count|avg_monthly_spend|churn_rate|
+----------+--------------+-----------------+----------+
|Technology|           114|           177.76|       0.2|
| Education|           145|           176.99|      0.19|
|  Business|           123|           184.05|      0.16|
|Healthcare|           118|           173.84|      0.14|
+----------+--------------+-----------------+----------+



### Task 7: Identify high-risk customers



In [0]:
spark.sql("""
    SELECT
        CUSTOMER_ID AS Customers,
        visits_per_month,
        support_calls,
        ROUND(monthly_spend, 0) AS monthly_spend
    FROM customers
    WHERE visits_per_month <= 3
      AND support_calls >= 4
      AND monthly_spend < 150
""").show()


+---------+----------------+-------------+-------------+
|Customers|visits_per_month|support_calls|monthly_spend|
+---------+----------------+-------------+-------------+
|       53|               3|            5|        119.0|
|       63|               1|            5|         95.0|
|      138|               3|            6|         15.0|
|      216|               2|            4|         52.0|
|      273|               3|            4|         24.0|
|      335|               3|            6|        121.0|
|      400|               1|            5|        112.0|
+---------+----------------+-------------+-------------+



### Task 8: Interpretation


In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator


### Task 9: Create the feature vector


In [0]:
feature_columns = ["age","annual_income","monthly_spend","visits_per_month","years_as_customer","support_calls","discount_used"]
# TODO: Assemble features and create modeling_df with features and label
modeling_df = VectorAssembler(inputCols=feature_columns, outputCol="features").transform(customers_df).select("features", "churned")


### Task 10: Split the data 70/30 using seed 42


In [0]:

(training_df, test_df) = modeling_df.randomSplit([.7, .3], seed=42)



### Task 11: Train logistic regression with maxIter=20


In [0]:

lr = LogisticRegression(labelCol="churned", maxIter=20)
lr_model = lr.fit(training_df)

### Task 12: Generate predictions and display label, prediction, probability


In [0]:

predictions = lr_model.transform(test_df)
print(predictions.select("features", "churned", "prediction", "probability").show())



+--------------------+-------+----------+--------------------+
|            features|churned|prediction|         probability|
+--------------------+-------+----------+--------------------+
|[18.0,56981.0,235...|      0|       0.0|[0.91292322574795...|
|[18.0,89100.0,280...|      0|       0.0|[0.90493185463157...|
|[19.0,55512.0,234...|      0|       0.0|[0.92091558529330...|
|[19.0,56871.0,204...|      1|       0.0|[0.85357105278807...|
|[20.0,69970.0,170...|      1|       0.0|[0.83109286857098...|
|[20.0,71840.0,133...|      0|       0.0|[0.86057489688126...|
|[20.0,84124.0,178...|      0|       0.0|[0.83869627551673...|
|[21.0,61318.0,349...|      1|       0.0|[0.92642902032511...|
|[21.0,75009.0,90....|      0|       0.0|[0.79413355591944...|
|[21.0,113641.0,14...|      0|       0.0|[0.91777892435563...|
|[22.0,18000.0,127...|      0|       0.0|[0.77994990081405...|
|[22.0,41913.0,177...|      0|       0.0|[0.87762856589528...|
|[22.0,45712.0,258...|      0|       0.0|[0.83822167216

### Task 13: Calculate AUC


In [0]:

evaluator = BinaryClassificationEvaluator( labelCol="churned")
print("AUC: {0:.4f}".format( evaluator.evaluate(predictions) ))


AUC: 0.6499


### Task 14: Calculate accuracy


In [0]:

accuracy = 0.0
for row in predictions.select("prediction", "churned").collect():
  if row.prediction == row.churned:
    accuracy += 1.0
print("Accuracy: {0:.4f}".format(accuracy / predictions.count()))
lr_model.coefficients

Accuracy: 0.8016


DenseVector([0.0049, -0.0, -0.0048, -0.0487, -0.0128, 0.0959, -0.2368])